# Building a RAG application from scratch

Here is a high-level overview of the system we want to build:

<img src='images/system1.png' width="1200">

In [1]:
pip install -r requirements.txt

  Using cached langchain-1.4.7-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_google_genai-4.4.2-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_pinecone-0.0.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached docarray-0.41.0-py3-none-any.whl.metadata (36 kB)
  Using cached pinecone_client-6.0.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached ruff-0.17.3-py3-none-macosx_11_0_arm64.whl.metadata (20 kB)
  Using cached langchain_core-1.7.3-py3-none-any.whl.metadata (4.8 kB)
  Using cached langgraph-1.2.19-py3-none-any.whl.metadata (4.9 kB)
INFO: pip is looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain-1.4.6-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.2-py3-none-any.whl.metadata (6.2 kB)
INFO: pip is still looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.


Let's start by loading the environment variables we need to use.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")


## Setting up the model
Let's define the LLM model that we'll use as part of the workflow. In this version we'll use Google's Gemini through the [langchain-google-genai](https://python.langchain.com/docs/integrations/chat/google_generative_ai/) package.


In [9]:
# Install the required package if it's missing in the environment

from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", google_api_key=GOOGLE_API_KEY)


We can test the model by asking a simple question.

In [10]:
model.invoke("What MLB team won the World Series during the COVID-19 pandemic?")

AIMessage(content=[{'type': 'text', 'text': 'The **Los Angeles Dodgers** won the World Series during the 2020 COVID-19 pandemic, defeating the Tampa Bay Rays four games to two.', 'extras': {'signature': 'EnMKcQFpFH0TtCuVYxyA1MnL0yVNF8ImDJm4psKBTewEtURHvyVTK5zqM96pceDZxClADh8zSqcajr9/t3nyy6hYeM6sAMVYE4zRWV2Wo1SMMcfmBpTS/BQHYDqjEk/1bNuTX+SAzeaEa8JWdbSURXsot2hx'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a19ccf-51c6-7405-b35c-e1337415e38a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 33, 'total_tokens': 49, 'input_token_details': {'cache_read': 0}})

The result from the model is an `AIMessage` instance containing the answer. We can extract this answer by chaining the model with an [output parser](https://python.langchain.com/docs/how_to/#output-parsers).

Here is what chaining the model with an output parser looks like:

<img src='images/chain1.png' width="1200">

For this example, we'll use a simple `StrOutputParser` to extract the answer as a string.

In [11]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

chain = model | parser # this mean that the output of the model will be passed to the parser using the pipe operator
chain.invoke("What MLB team won the World Series during the COVID-19 pandemic?")

'The **Los Angeles Dodgers** won the World Series during the 2020 COVID-19 pandemic, defeating the Tampa Bay Rays four games to two.'

## Introducing prompt templates

We want to provide the model with some context and the question. [Prompt templates](https://python.langchain.com/docs/how_to/#prompt-templates) are a simple way to define and reuse prompts.

In [13]:
from langchain_core.prompts import ChatPromptTemplate

template = """
Answer the question based on the context below. If you can't 
answer the question, reply "I don't know".

Context: {context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt.format(context="Mary's sister is Susana", question="Who is Mary's sister?")

'Human: \nAnswer the question based on the context below. If you can\'t \nanswer the question, reply "I don\'t know".\n\nContext: Mary\'s sister is Susana\n\nQuestion: Who is Mary\'s sister?\n'

We can now chain the prompt with the model and the output parser.

<img src='images/chain2.png' width="1200">

In [21]:
chain = prompt | model | parser

## Combining chains

We can combine different chains to create more complex workflows. For example, let's create a second chain that translates the answer from the first chain into a different language.

Let's start by creating a new prompt template for the translation chain:

In [14]:
translation_prompt = ChatPromptTemplate.from_template(
    "Translate {answer} to {language}"
)

We can now create a new translation chain that combines the result from the first chain with the translation prompt.

Here is what the new workflow looks like:

<img src='images/chain3.png' width="1200">

In [ ]:
from operator import itemgetter


translation_chain = (
    {"answer": chain, "language": itemgetter("language")} | translation_prompt | model | parser
)
#itemgetter is a function that returns a callable that fetches the given item(s) from its operand. In this case, it will fetch the value of the "language" key from the input dictionary.
translation_chain.invoke(
    {
        "context": "Mary's sister is Susana. She doesn't have any more siblings.",
        "question": "How many sisters does Mary have?",
        "language": "Spanish",
    }
)

'María tiene una hermana, Susana.'

## Fetching the YouTube transcript

The context we want to send the model comes from a YouTube video. Instead of downloading the audio and transcribing it with a model like Whisper, we can fetch the video's transcript directly from YouTube using [youtube-transcript-api](https://github.com/jdepoix/youtube-transcript-api). YouTube already stores transcripts (human-made or auto-generated) for most videos, so we get the same text for free.


In [15]:
%pip install youtube-transcript-api


  Using cached youtube_transcript_api-1.2.4-py3-none-any.whl.metadata (24 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
Using cached defusedxml-0.7.1-py2.py3-none-any.whl (25 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [youtube-transcript-api]

[notice] A new release of pip is available: 26.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
from youtube_transcript_api import YouTubeTranscriptApi

# Extract the video ID from the URL.
YOUTUBE_VIDEO = "https://www.youtube.com/watch?v=BrsocJb-fAo"
VIDEO_ID = YOUTUBE_VIDEO.split("v=", 1)[1].split("&", 1)[0]

# We only do this if we haven't created the transcription file yet.
if not os.path.exists("transcription.txt"):
    transcript = YouTubeTranscriptApi().fetch(VIDEO_ID, languages=["en"])

    with open("transcription.txt", "w") as file:
        # Join the timestamped snippets into a single block of text.
        file.write(" ".join(snippet.text for snippet in transcript))


Let's read the transcription and display the first few characters to ensure everything works as expected.

In [19]:
with open("transcription.txt") as file:
    transcription = file.read()

transcription[:100]

'Hi, so today I want to show you one of the hottest skills in 2024, which is how to build retrieval a'

## Using the entire transcription as context

If we try to invoke the chain using the transcription as context, the model may return an error because the context is too long.

Large Language Models support limited context sizes. Gemini models have very large context windows, so this particular video may fit — but it's still inefficient and doesn't scale to longer videos or models with smaller context windows. We need a different solution.


In [24]:
try:

    print(chain.invoke({
        "context": transcription,
        "question": "what is the main topic of the video?",
    }))
except Exception as e:
    print(e)

The main topic of the video is how to build a Retrieval Augmented Generation (RAG) system using Python, the OpenAI API (specifically the GPT-3.5 Turbo model), and the LangChain framework. The instructor demonstrates this by building an application that allows users to ask questions based on the content of a specific YouTube video transcript.


## Splitting the transcription

Since we can't use the entire transcription as the context for the model, a potential solution is to split the transcription into smaller chunks. We can then invoke the model using only the relevant chunks to answer a particular question:

<img src='images/system2.png' width="1200">

Let's start by loading the transcription in memory:

In [25]:
%pip install -q langchain-community


[notice] A new release of pip is available: 26.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [26]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("transcription.txt")
text_documents = loader.load()
text_documents

/var/folders/hg/l1pbv_nd6wx50_wkjv7lgp6m0000gn/T/ipykernel_74526/1486996460.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': 'transcription.txt'}, page_content='Hi, so today I want to show you one of the hottest skills in 2024, which is how to build retrieval augmented generation systems using Python, using the OpenAI API. Specifically, we\'re going to be using the GPT 3.5 Turbo model and how to use LangChain. And the reason I I picked LangChain is because it\'s by far the most popular framework to build LLM applications. Now, by the end of this video, I I I not only want you to understand how to put together the code, but I want to explain, I want you to leave understanding the reasoning behind every single component that we use to build the solution. So, let me go here. Let me show you really quick what you are going to find. I\'m going to be using Visual Studio Code throughout this video to run the code. The code is in a in a Jupyter notebook. Now, I\'m using the extension, the Jupyter extension within Visual Studio Code to run my code within Visual Studio, but if you want to

There are many different ways to split a document. For this example, we'll use a simple splitter that splits the document into chunks of a fixed size. Check [Text Splitters](https://python.langchain.com/docs/how_to/#text-splitters) for more information about different approaches to splitting documents.

For illustration purposes, let's split the transcription into chunks of 100 characters with an overlap of 20 characters and display the first few chunks:

In [ ]:
# If not already installed:
# %pip install -q langchain-text-splitters

from langchain_text_splitters import RecursiveCharacterTextSplitter
#recursive character text splitter is a class that splits text into chunks of a specified size, with a specified overlap between chunks. This is useful for processing large documents in smaller pieces, which can be more manageable for certain applications, such as natural language processing or machine learning tasks.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
text_splitter.split_documents(text_documents)[:5]

[Document(metadata={'source': 'transcription.txt'}, page_content='Hi, so today I want to show you one of the hottest skills in 2024, which is how to build retrieval'),
 Document(metadata={'source': 'transcription.txt'}, page_content='to build retrieval augmented generation systems using Python, using the OpenAI API. Specifically,'),
 Document(metadata={'source': 'transcription.txt'}, page_content="API. Specifically, we're going to be using the GPT 3.5 Turbo model and how to use LangChain. And"),
 Document(metadata={'source': 'transcription.txt'}, page_content="use LangChain. And the reason I I picked LangChain is because it's by far the most popular"),
 Document(metadata={'source': 'transcription.txt'}, page_content='the most popular framework to build LLM applications. Now, by the end of this video, I I I not only')]

For our specific application, let's use 1000 characters instead:

In [28]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
documents = text_splitter.split_documents(text_documents)

## Finding the relevant chunks

Given a particular question, we need to find the relevant chunks from the transcription to send to the model. Here is where the idea of **embeddings** comes into play.

An embedding is a mathematical representation of the semantic meaning of a word, sentence, or document. It's a projection of a concept in a high-dimensional space. Embeddings have a simple characteristic: The projection of related concepts will be close to each other, while concepts with different meanings will lie far away. You can use the [Cohere's Embed Playground](https://dashboard.cohere.com/playground/embed) to visualize embeddings in two dimensions.

To provide with the most relevant chunks, we can use the embeddings of the question and the chunks of the transcription to compute the similarity between them. We can then select the chunks with the highest similarity to the question and use them as the context for the model:

<img src='images/system3.png' width="1200">

Let's generate embeddings for an arbitrary query:

In [29]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# text-embedding-004 generates 768-dimensional embeddings.
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
embedded_query = embeddings.embed_query("Who is Mary's sister?")

print(f"Embedding length: {len(embedded_query)}")
print(embedded_query[:10])


GoogleGenerativeAIError: Error embedding content (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

To illustrate how embeddings work, let's first generate the embeddings for two different sentences:

In [36]:
sentence1 = embeddings.embed_query("Mary's sister is Susana")
sentence2 = embeddings.embed_query("Pedro's mother is a teacher")

We can now compute the similarity between the query and each of the two sentences. The closer the embeddings are, the more similar the sentences will be.

We can use [Cosine Similarity](https://en.wikipedia.org/wiki/Cosine_similarity) to calculate the similarity between the query and each of the sentences:

In [37]:
from sklearn.metrics.pairwise import cosine_similarity

query_sentence1_similarity = cosine_similarity([embedded_query], [sentence1])[0][0]
query_sentence2_similarity = cosine_similarity([embedded_query], [sentence2])[0][0]

query_sentence1_similarity, query_sentence2_similarity

(0.9172681467301312, 0.7680251090423726)

## Setting up a Vector Store

We need an efficient way to store document chunks, their embeddings, and perform similarity searches at scale. To do this, we'll use a **vector store**.

A vector store is a database of embeddings that specializes in fast similarity searches. 

<img src='images/system4.png' width="1200">

To understand how a vector store works, let's create one in memory and add a few embeddings to it:

In [38]:
from langchain_community.vectorstores import DocArrayInMemorySearch

vectorstore1 = DocArrayInMemorySearch.from_texts(
    [
        "Mary's sister is Susana",
        "John and Tommy are brothers",
        "Patricia likes white cars",
        "Pedro's mother is a teacher",
        "Lucia drives an Audi",
        "Mary has two siblings",
    ],
    embedding=embeddings,
)

We can now query the vector store to find the most similar embeddings to a given query:

In [39]:
vectorstore1.similarity_search_with_score(query="Who is Mary's sister?", k=3)

[(Document(page_content="Mary's sister is Susana"), 0.9172681550033158),
 (Document(page_content='Mary has two siblings'), 0.9045628481161789),
 (Document(page_content='John and Tommy are brothers'), 0.8015500435454899)]

## Connecting the vector store to the chain

We can use the vector store to find the most relevant chunks from the transcription to send to the model. Here is how we can connect the vector store to the chain:

<img src='images/chain4.png' width="1200">

We need to configure a [Retriever](https://python.langchain.com/docs/how_to/#retrievers). The retriever will run a similarity search in the vector store and return the most similar documents back to the next step in the chain.

We can get a retriever directly from the vector store we created before: 

In [40]:
retriever1 = vectorstore1.as_retriever()
retriever1.invoke("Who is Mary's sister?")

[Document(page_content="Mary's sister is Susana"),
 Document(page_content='Mary has two siblings'),
 Document(page_content='John and Tommy are brothers'),
 Document(page_content="Pedro's mother is a teacher")]

Our prompt expects two parameters, "context" and "question." We can use the retriever to find the chunks we'll use as the context to answer the question.

We can create a map with the two inputs by using the [`RunnableParallel`](https://python.langchain.com/docs/how_to/parallel/) and [`RunnablePassthrough`](https://python.langchain.com/docs/how_to/passthrough/) classes. This will allow us to pass the context and question to the prompt as a map with the keys "context" and "question."

In [41]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

setup = RunnableParallel(context=retriever1, question=RunnablePassthrough())
setup.invoke("What color is Patricia's car?")

{'context': [Document(page_content='Patricia likes white cars'),
  Document(page_content='Lucia drives an Audi'),
  Document(page_content="Pedro's mother is a teacher"),
  Document(page_content="Mary's sister is Susana")],
 'question': "What color is Patricia's car?"}

Let's now add the setup map to the chain and run it:



In [42]:
chain = setup | prompt | model | parser
chain.invoke("What color is Patricia's car?")

'White'

Let's invoke the chain using another example:

In [43]:
chain.invoke("What car does Lucia drive?")

'Lucia drives an Audi.'

## Loading transcription into the vector store

We initialized the vector store with a few random strings. Let's create a new vector store using the chunks from the video transcription.

In [44]:
vectorstore2 = DocArrayInMemorySearch.from_documents(documents, embeddings)

Let's set up a new chain using the correct vector store. This time we are using a different equivalent syntax to specify the [`RunnableParallel`](https://python.langchain.com/docs/how_to/parallel/) portion of the chain:

In [45]:
chain = (
    {"context": vectorstore2.as_retriever(), "question": RunnablePassthrough()}
    | prompt
    | model
    | parser
)
chain.invoke("What is synthetic intelligence?")

'Synthetic intelligence is described as the next stage of development in the context provided. It is mentioned that synthetic intelligences will uncover puzzles in the universe and solve them.'

## Setting up Pinecone

So far we've used an in-memory vector store. In practice, we need a vector store that can handle large amounts of data and perform similarity searches at scale. For this example, we'll use [Pinecone](https://www.pinecone.io/).

The first step is to create a Pinecone account, set up an index, get an API key, and set it as an environment variable `PINECONE_API_KEY`.

Then, we can load the transcription documents into Pinecone:

In [46]:
from langchain_pinecone import PineconeVectorStore

index_name = "youtube-rag-index"

pinecone = PineconeVectorStore.from_documents(
    documents, embeddings, index_name=index_name
)

/Users/svpino/dev/youtube-rag/.venv/lib/python3.9/site-packages/pinecone/data/index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


Let's now run a similarity search on a pinecone to make sure everything works:

In [47]:
pinecone.similarity_search("What is Hollywood going to start doing?")[:3]

[Document(page_content="It's like high quality audio and you're speaking usually pretty clearly. I don't know what open AI's plans are either. Yeah, there's always fun projects basically. And stable diffusion also is opening up a huge amount of experimentation. I would say in the visual realm and generating images and videos and movies. I'll think like videos now. And so that's going to be pretty crazy. That's going to almost certainly work and it's going to be really interesting when the cost of content creation is going to fall to zero. You used to need a painter for a few months to paint a thing and now it's going to be speak to your phone to get your video. So Hollywood will start using it to generate scenes, which completely opens up. Yeah, so you can make a movie like Avatar eventually for under a million dollars. Much less. Maybe just by talking to your phone. I mean, I know it sounds kind of crazy. And then there'd be some voting mechanism. Like how do you have a, like, would t

Let's setup the new chain using Pinecone as the vector store:

In [48]:
chain = (
    {"context": pinecone.as_retriever(), "question": RunnablePassthrough()}
    | prompt
    | model
    | parser
)

chain.invoke("What is Hollywood going to start doing?")

'Hollywood is going to start using AI to generate scenes for movies.'